In [ ]:
import warnings
from dataclasses import dataclass, asdict

import numpy as np
import matplotlib.pyplot as plt
import qnm

warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

# Physical constants
G = 6.67430e-11
C = 299_792_458.0
MSUN = 1.988409870698051e30

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

@dataclass
class KerrRemnant:
    mass_msun: float
    chi: float

    def __post_init__(self):
        if self.mass_msun <= 0:
            raise ValueError("mass_msun must be positive")
        if not (-1.0 < self.chi < 1.0):
            raise ValueError("chi must satisfy -1 < chi < 1")

    @property
    def mass_si(self) -> float:
        return self.mass_msun * MSUN

    @property
    def time_unit(self) -> float:
        """Geometric mass scale GM/c^3 in seconds."""
        return G * self.mass_si / C**3

def get_kerr_qnm(remnant: KerrRemnant, l: int, m: int, n: int, s: int = -2) -> dict:
    """
    Query a Kerr quasinormal mode from qnm and convert to physical units.
    """
    mode_seq = qnm.modes_cache(s=s, l=l, m=m, n=n)
    result = mode_seq(a=remnant.chi)
    omega = result[0]

    omega_r_geom = float(np.real(omega))
    omega_i_geom = float(-np.imag(omega))  # qnm uses omega_R - i omega_I

    M_sec = remnant.time_unit
    omega_r_si = omega_r_geom / M_sec
    omega_i_si = omega_i_geom / M_sec

    return {
        "l": l,
        "m": m,
        "n": n,
        "omega_complex_geom": omega,
        "omega_r_geom": omega_r_geom,
        "omega_i_geom": omega_i_geom,
        "omega_r_si": omega_r_si,
        "omega_i_si": omega_i_si,
        "f_hz": omega_r_si / (2.0 * np.pi),
        "tau_s": 1.0 / omega_i_si,
    }

remnant = KerrRemnant(mass_msun=60.0, chi=0.7)

for (l, m, n) in [(2, 2, 0), (2, 2, 1), (3, 3, 0)]:
    q = get_kerr_qnm(remnant, l, m, n)
    print(
        f"({l},{m},{n})  "
        f"Mω = {q['omega_complex_geom']:.6f}   "
        f"f = {q['f_hz']:.2f} Hz   "
        f"tau = {1e3*q['tau_s']:.3f} ms"
    )

@dataclass
class RingdownMode:
    l: int
    m: int
    n: int
    amplitude: float
    phase: float


@dataclass
class RingdownSignal:
    t: np.ndarray
    h_complex: np.ndarray

    @property
    def h_plus(self) -> np.ndarray:
        return np.real(self.h_complex)

    @property
    def h_cross(self) -> np.ndarray:
        return -np.imag(self.h_complex)

def component_strain_qnm(
    remnant: KerrRemnant,
    mode: RingdownMode,
    t: np.ndarray,
    t0: float = 0.0,
    s: int = -2,
) -> np.ndarray:
    """
    Build one complex damped Kerr QNM component.
    """
    q = get_kerr_qnm(remnant, mode.l, mode.m, mode.n, s=s)

    omega_r = q["omega_r_si"]
    omega_i = q["omega_i_si"]

    dt = t - t0
    mask = dt >= 0.0

    h = np.zeros_like(t, dtype=np.complex128)
    h[mask] = (
        mode.amplitude
        * np.exp(-omega_i * dt[mask])
        * np.exp(-1j * (omega_r * dt[mask] + mode.phase))
    )
    return h


def generate_gr_ringdown(
    remnant: KerrRemnant,
    modes: list[RingdownMode],
    duration: float,
    sample_rate: float,
    t0: float = 0.0,
    s: int = -2,
) -> RingdownSignal:
    """
    Sum several qnm-based Kerr modes into one GR ringdown signal.
    """
    if duration <= 0:
        raise ValueError("duration must be positive")
    if sample_rate <= 0:
        raise ValueError("sample_rate must be positive")

    t = np.arange(0.0, duration, 1.0 / sample_rate)
    h = np.zeros_like(t, dtype=np.complex128)

    for mode in modes:
        h += component_strain_qnm(remnant, mode, t, t0=t0, s=s)

    return RingdownSignal(t=t, h_complex=h)

remnant = KerrRemnant(mass_msun=60.0, chi=0.7)

modes = [
    RingdownMode(l=2, m=2, n=0, amplitude=1.00, phase=0.0),
    RingdownMode(l=2, m=2, n=1, amplitude=0.45, phase=0.3),
    RingdownMode(l=3, m=3, n=0, amplitude=0.25, phase=-0.5),
]

duration = 0.05
sample_rate = 8192
t0 = 0.002

signal_gr = generate_gr_ringdown(
    remnant=remnant,
    modes=modes,
    duration=duration,
    sample_rate=sample_rate,
    t0=t0,
)

plt.figure()
plt.plot(signal_gr.t, signal_gr.h_plus, label=r"$h_+$")
plt.plot(signal_gr.t, signal_gr.h_cross, label=r"$h_\times$")
plt.plot(signal_gr.t, np.abs(signal_gr.h_complex), "--", label="envelope")
plt.xlabel("Time [s]")
plt.ylabel("Strain [arb.]")
plt.title("Multimode Kerr ringdown template in GR")
plt.legend()
plt.show()

plt.figure()

for mode in modes:
    sig = generate_gr_ringdown(
        remnant=remnant,
        modes=[mode],
        duration=duration,
        sample_rate=sample_rate,
        t0=t0,
    )
    plt.plot(sig.t, sig.h_plus, label=f"({mode.l},{mode.m},{mode.n})")

plt.xlabel("Time [s]")
plt.ylabel(r"$h_+$ [arb.]")
plt.title("Individual Kerr QNM contributions")
plt.legend()
plt.show()

mode_table = []

for mode in modes:
    q = get_kerr_qnm(remnant, mode.l, mode.m, mode.n)
    mode_table.append({
        "mode": f"({mode.l},{mode.m},{mode.n})",
        "amplitude": mode.amplitude,
        "phase": mode.phase,
        "omega_r_geom": q["omega_r_geom"],
        "omega_i_geom": q["omega_i_geom"],
        "f_hz": q["f_hz"],
        "tau_ms": 1e3 * q["tau_s"],
    })

mode_table

def collect_mode_vs_spin(
    mass_msun: float,
    spins: np.ndarray,
    mode_list: list[tuple[int, int, int]],
):
    data = {}

    for (l, m, n) in mode_list:
        freqs = []
        taus = []
        for chi in spins:
            rem = KerrRemnant(mass_msun=mass_msun, chi=float(chi))
            q = get_kerr_qnm(rem, l, m, n)
            freqs.append(q["f_hz"])
            taus.append(q["tau_s"])
        data[(l, m, n)] = {
            "f_hz": np.array(freqs),
            "tau_s": np.array(taus),
        }

    return data

spins = np.linspace(0.05, 0.98, 80)
mode_list = [(2, 2, 0), (2, 2, 1), (3, 3, 0)]

spin_data = collect_mode_vs_spin(
    mass_msun=60.0,
    spins=spins,
    mode_list=mode_list,
)

plt.figure()

for key in mode_list:
    plt.plot(spins, spin_data[key]["f_hz"], label=f"{key}")

plt.xlabel(r"Final spin $\chi$")
plt.ylabel("Frequency [Hz]")
plt.title("Kerr QNM frequency vs spin")
plt.legend(title="Mode")
plt.show()

plt.figure()

for key in mode_list:
    plt.plot(spins, 1e3 * spin_data[key]["tau_s"], label=f"{key}")

plt.xlabel(r"Final spin $\chi$")
plt.ylabel("Damping time [ms]")
plt.title("Kerr QNM damping time vs spin")
plt.legend(title="Mode")
plt.show()

masses = np.linspace(20.0, 120.0, 60)
fixed_spin = 0.7

mass_freqs = []
mass_taus = []

for M in masses:
    rem = KerrRemnant(mass_msun=float(M), chi=fixed_spin)
    q = get_kerr_qnm(rem, 2, 2, 0)
    mass_freqs.append(q["f_hz"])
    mass_taus.append(q["tau_s"])

mass_freqs = np.array(mass_freqs)
mass_taus = np.array(mass_taus)

plt.figure()
plt.plot(masses, mass_freqs)
plt.xlabel(r"Final mass [$M_\odot$]")
plt.ylabel(r"$f_{220}$ [Hz]")
plt.title(r"Mass scaling of $f_{220}$ at fixed spin")
plt.show()

plt.figure()
plt.plot(masses, 1e3 * mass_taus)
plt.xlabel(r"Final mass [$M_\odot$]")
plt.ylabel(r"$\tau_{220}$ [ms]")
plt.title(r"Mass scaling of $\tau_{220}$ at fixed spin")
plt.show()

def waveform_inner_product(h1: np.ndarray, h2: np.ndarray, dt: float) -> float:
    return float(np.sum(h1 * h2) * dt)

def waveform_norm(h: np.ndarray, dt: float) -> float:
    return np.sqrt(max(waveform_inner_product(h, h, dt), 0.0))

def waveform_overlap(h1: np.ndarray, h2: np.ndarray, dt: float) -> float:
    n1 = waveform_norm(h1, dt)
    n2 = waveform_norm(h2, dt)
    if n1 == 0.0 or n2 == 0.0:
        return 0.0
    return waveform_inner_product(h1, h2, dt) / (n1 * n2)

def waveform_mismatch(h1: np.ndarray, h2: np.ndarray, dt: float) -> float:
    return 1.0 - waveform_overlap(h1, h2, dt)

signal_220 = generate_gr_ringdown(
    remnant=remnant,
    modes=[RingdownMode(2, 2, 0, 1.0, 0.0)],
    duration=duration,
    sample_rate=sample_rate,
    t0=t0,
)

dt = signal_gr.t[1] - signal_gr.t[0]
ov = waveform_overlap(signal_220.h_plus, signal_gr.h_plus, dt)
mm = waveform_mismatch(signal_220.h_plus, signal_gr.h_plus, dt)

print(f"Overlap(single 220, multimode) = {ov:.6f}")
print(f"Mismatch(single 220, multimode) = {mm:.6f}")

export_data = {
    "remnant": asdict(remnant),
    "modes": [asdict(m) for m in modes],
    "mode_table": mode_table,
    "t": signal_gr.t,
    "h_plus": signal_gr.h_plus,
    "h_cross": signal_gr.h_cross,
    "h_complex": signal_gr.h_complex,
}

np.savez(
    "gr_ringdown_template_qnm.npz",
    t=signal_gr.t,
    h_plus=signal_gr.h_plus,
    h_cross=signal_gr.h_cross,
    h_complex=signal_gr.h_complex,
)

export_data["remnant"], export_data["modes"][:2], export_data["mode_table"]